# BoFM EModE parser v2 — MacBERTh transformer (the GPU keystone bet)

Trains a **biaffine + MacBERTh** (historical-English BERT) dependency parser on the **fixed-converter,
author-held-out** PCEEC→UD data, then predicts the held-out test set + re-parses the Book of Mormon.
This is the GPU re-try after the CPU parser lost to Stanza 21–6 on the blind gate. Claude scores
per-label F + runs the same blind head-to-head vs Stanza when you return the two output files.

**Runtime → Change runtime type → GPU (T4 ok).** Run all. Upload `train-package-v2.zip` at cell 2.

In [ ]:
# 1. GPU + install
!nvidia-smi -L
!pip install -q supar==1.1.4 'transformers>=4.20,<4.40'
import supar, torch; print('supar', supar.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2. Upload train-package-v2.zip  (train2/dev2/test2.conllu + bofm_toparse.conllu)
from google.colab import files
import zipfile, os
up = files.upload()
with zipfile.ZipFile(next(iter(up))) as z: z.extractall('.')
print(sorted(f for f in os.listdir('.') if f.endswith('.conllu')))

In [ ]:
# 3. Train biaffine + MacBERTh (historical-English encoder). ~30-50 min on a T4.
#    Author-held-out split; fixed converter (clause-type error 6.6%).
!python -m supar.cmds.biaffine_dep train -b -d 0     -p bofm-macberth.parser -f bert --bert emanjavacas/MacBERTh     --train train2.conllu --dev dev2.conllu --test test2.conllu     --batch-size 1000 --epochs 10

In [ ]:
# 4. Held-out test LAS/UAS (honest, author-held-out)
!python -m supar.cmds.biaffine_dep evaluate -d 0 -p bofm-macberth.parser --data test2.conllu

In [ ]:
# 5a. Predict the held-out TEST (Claude scores per-label F vs gold from this)
!python -m supar.cmds.biaffine_dep predict -d 0 -p bofm-macberth.parser     --data test2.conllu --pred test2_macberth.conllu
# 5b. Re-parse the Book of Mormon (Claude runs the blind gate vs Stanza from this)
!python -m supar.cmds.biaffine_dep predict -d 0 -p bofm-macberth.parser     --data bofm_toparse.conllu --pred bofm_macberth.conllu
print('done')

In [ ]:
# 6. Download the two prediction files (send BOTH back to Claude)
from google.colab import files
files.download('test2_macberth.conllu')
files.download('bofm_macberth.conllu')

## Send back to Claude
`test2_macberth.conllu` (→ per-label F on held-out authors) **and** `bofm_macberth.conllu`
(→ the blind head-to-head gate vs Stanza, same 30-sentence protocol the CPU parser failed 21–6).
If MacBERTh beats Stanza on the gate, the keystone is alive and we build TF v0.2; if not, we close
the parser track and BoFM rests on Stanza + v2-sprays.